<a href="https://colab.research.google.com/github/veeennn/Stationary_Classification/blob/main/Stationary_Classification_KNN_and_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#import
from google.colab import drive
drive.mount('/content/drive')
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn import datasets
from sklearn.model_selection import train_test_split

Mounted at /content/drive


Gaussian & Otsu Preprocess

In [ ]:
preprocessedImages = []
preprocessedLabels = []
lower = np.array([160, 160, 160], dtype=np.uint8)
upper = np.array([255, 255, 255], dtype=np.uint8)

image_folder = '/content/drive/MyDrive/Project_Dip'
for class_name in os.listdir(image_folder):
  class_dir = os.path.join(image_folder, class_name)
  for image_name in os.listdir(class_dir):
      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue

      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      blurred = cv2.GaussianBlur(gray, (5, 5), 0)

      blurred = cv2.resize(blurred, (64,64), interpolation = cv2.INTER_AREA)
      otsu = threshold_otsu(blurred)
      result = (gray > otsu)

      preprocessedImages.append(result)
      preprocessedLabels.append(class_name)

preprocessedImages = np.array(preprocessedImages)
preprocessedLabels = np.array(preprocessedLabels)

for i in range(len(preprocessedImages)):
  plt.imshow(preprocessedImages[i], cmap='gray')
  plt.title(preprocessedLabels[i])
  plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Project_Dip'

Rotate & Resize Augmentation

In [ ]:
train_data = []
train_labels = []

image_folder = '/content/drive/MyDrive/DATASET_PROJECT_DIP'
train_dir = os.path.join(image_folder, "training_set")
for class_name in os.listdir(train_dir):
  class_dir = os.path.join(train_dir, class_name)
  for image_name in os.listdir(class_dir):
      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      gray = cv2.GaussianBlur(gray, (3,3), 0)


      laplacian = cv2.Laplacian(gray,cv2.CV_64F, ksize = 3)
      enchanced_img = cv2.convertScaleAbs(gray - laplacian)
      train_data.append(enchanced_img.flatten())
      train_labels.append(class_name)

train_data = np.array(train_data)
train_labels = np.array(train_labels)
for i in range(len(train_data)):
  plt.imshow(train_data[i].reshape(64, 64), cmap='gray')
  plt.title(train_labels[i])
  plt.show()

In [ ]:
augmentedImages = []
augmentedLabels = []
image_folder = '/content/drive/MyDrive/DATASET_PROJECT_DIP'
for class_name in os.listdir(image_folder):
  class_dir = os.path.join(image_folder, class_name)
  augmented_class_dir = os.path.join('/content/drive/MyDrive/DATASET_PROJECT_DIP/augmented', class_name)

  for image_name in os.listdir(class_dir):
      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue

      image = cv2.resize(image, (64,64))
      (h, w) = image.shape[:2]
      center = (w / 2, h / 2)

      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(image, M, (w, h), borderValue=(255, 255, 255))

          base_name, ext = os.path.splitext(image_name)
          new_image_name = f"{base_name}_rotated_{angle}{ext}"
          save_path = os.path.join(augmented_class_dir, new_image_name)

          augmentedImages.append(rotated_image)
          augmentedLabels.append(class_name)

          # cv2.imwrite(save_path, rotated_image)

augmentedImages = np.array(augmentedImages)
augmentedLabels = np.array(augmentedLabels)

for i in range(len(augmentedImages)):
  plt.imshow(augmentedImages[i])
  plt.title(augmentedLabels[i])
  plt.show()

Canny Preprocess

In [ ]:
preprocessedImages = []
preprocessedLabels = []
image_folder = '/content/drive/MyDrive/DATASET_PROJECT_DIP/training_set'
for class_name in os.listdir(image_folder):
  class_dir = os.path.join(image_folder, class_name)
  for image_name in os.listdir(class_dir):
      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

      img_canny = cv2.Canny(gray, 100, 200)
      (h, w) = img_canny.shape[:2]
      center = (w / 2, h / 2)
      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(img_canny, M, (w, h),borderValue=(0, 0, 0))
          preprocessedImages.append(rotated_image.flatten())
          preprocessedLabels.append(class_name)
preprocessedImages = np.array(preprocessedImages)
preprocessedLabels = np.array(preprocessedLabels)

for i in range(len(preprocessedImages)):
  plt.imshow(preprocessedImages[i].reshape(64, 64), cmap='gray')
  plt.title(preprocessedLabels[i])
  plt.show()

Sobel Preprocess

In [ ]:
preprocessedImages = []
preprocessedLabels = []
image_folder = '/content/drive/MyDrive/DATASET_PROJECT_DIP/training_set'
for class_name in os.listdir(image_folder):
  class_dir = os.path.join(image_folder, class_name)
  for image_name in os.listdir(class_dir):
      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      img_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5)
      img_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)
      img_sobel = np.sqrt(img_x**2 + img_y**2)

      preprocessedImages.append(img_sobel)
      preprocessedLabels.append(class_name)

preprocessedImages = np.array(preprocessedImages)
preprocessedLabels = np.array(preprocessedLabels)

for i in range(len(preprocessedImages)):
  plt.imshow(preprocessedImages[i], cmap='gray')
  plt.title(preprocessedLabels[i])
  plt.show()


In [ ]:
#CANNY

from sklearn.preprocessing import LabelEncoder

train_data = []
train_labels = []

image_folder = '/content/drive/MyDrive/DATASET_PROJECT_DIP'
train_dir = os.path.join(image_folder, "training_set")
for class_name in os.listdir(train_dir):
  class_dir = os.path.join(train_dir, class_name)
  for image_name in os.listdir(class_dir):
      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      #canny
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      img_canny = cv2.Canny(gray, 100, 200)
      (h, w) = img_canny.shape[:2]
      center = (w / 2, h / 2)
      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(img_canny, M, (w, h), borderValue=(0,0,0))

          train_data.append(rotated_image.flatten())
          train_labels.append(class_name)

train_data = np.array(train_data)
train_labels = np.array(train_labels)


validation_data = []
validation_labels = []
validation_dir = os.path.join(image_folder, "validation_set")
for class_name in os.listdir(validation_dir):
  class_dir = os.path.join(validation_dir, class_name)
  for image_name in os.listdir(class_dir):

      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      #canny
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      img_canny = cv2.Canny(gray, 100, 200)
      (h, w) = img_canny.shape[:2]
      center = (w / 2, h / 2)
      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(img_canny, M, (w, h),borderValue=(0, 0, 0))
          validation_data.append(rotated_image.flatten())
          validation_labels.append(class_name)

validation_data = np.array(validation_data)
validation_labels = np.array(validation_labels)

le = LabelEncoder()
train_labels_encoded = le.fit_transform(train_labels)
validation_labels_encoded = le.transform(validation_labels)


knn_classifier = KNeighborsClassifier(n_neighbors = 3,metric='manhattan')
knn_classifier.fit(train_data, train_labels_encoded)
predictions = knn_classifier.predict(validation_data)
accuracy = accuracy_score(validation_labels_encoded, predictions)
print(f"Accuracy: {accuracy}")

#svm classifier
svm_classifier = svm.SVC(kernel="sigmoid")
svm_classifier.fit(train_data, train_labels)
predictions = svm_classifier.predict(validation_data)
accuracy = accuracy_score(validation_labels, predictions)
print(f"SVM Sigmoid Accuracy: {accuracy}")
svm_classifier = svm.SVC(kernel="linear")
svm_classifier.fit(train_data, train_labels)
predictions = svm_classifier.predict(validation_data)
accuracy = accuracy_score(validation_labels, predictions)
print(f"SVM Linear Accuracy: {accuracy}")


from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score, RepeatedStratifiedKFold, StratifiedKFold, LeaveOneOut
from sklearn.neural_network import MLPClassifier
from sklearn import svm
from sklearn.preprocessing import LabelEncoder
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os



#menggunakan gabungan train dan validation set
X = np.vstack((train_data, validation_data))
y = np.concatenate((train_labels, validation_labels))

# ubah label ke nilai numeric
le = LabelEncoder()
y_encoded = le.fit_transform(y)


# bagi data ke train dan test
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

# knn klassifier
knn = KNeighborsClassifier(n_neighbors= 2)
knn.fit(X_train, y_train)

# Make predictions
y_pred = knn.predict(X_test)

# evaluasi
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

#train validation
print("\nTrain Validation")
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  clf.fit(X_train, y_train)
  output=clf.predict(X_val)
  acc = 100*accuracy_score(y_val, output)
  print('k = %g, test accuracy = %g%%'%(k,acc))
clf = KNeighborsClassifier(n_neighbors=1)
clf.fit(X_train, y_train)
output = clf.predict(X_test)
acc = 100*accuracy_score(y_test, output)
print('k = %g, test accuracy = %g%%'%(1, acc))

#stratified repeated hold out
print("\nstratified repeated hold out")
sss = list(StratifiedShuffleSplit(n_splits=10, test_size=0.333, random_state=1).split(X, y_encoded))
mean_acc = []
acc = 0
K = 1
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=sss)
  mean_acc.append(100*np.mean(accuracy))
  if(100*accuracy.mean()>acc):
    acc = 100*accuracy.mean()
    K = k
print('highest mean accuracy is at k = %g, test accuracy = %g%%'%(K,acc))
plt.plot(range(1,11),mean_acc,'o-')
plt.title('KNN Accuracy with Stratified Repeated Hold Out')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Mean Accuracy (%)')
plt.show()

#10-fold cross validation
print("\n 10-fold cross validation")
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=10)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

#10-fold stratified validation
print("\n stratified validation")
skf = StratifiedKFold(n_splits=10)
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=skf)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

# leave one out
print("\n leave one out")
loo = LeaveOneOut()
for k in range(1,11):
  clf = KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=loo)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

# Evaluate other classifiers
ann = MLPClassifier(max_iter=1000, random_state=42)
svm_model = svm.SVC(random_state=42) # Instantiate SVM classifier

# 1. Train-test split
print("\nTrain-Test Split")
xtrain, xtest, ytrain, ytest = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    model.fit(xtrain, ytrain)
    ypred = model.predict(xtest)
    acc = accuracy_score(ytest, ypred)
    print(f"{name} Accuracy: {acc:.4f}")

# 2. Stratified repeated
print("\n Stratified Repeated")
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=rskf)
    print(f"{name}  Accuracy: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# 3.  Cross Validation
print("\n cross")
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=10)
    print(f"{name}  Accuracy: {np.mean(scores):.4f}")

# 4. Stratified
print("\n Stratified")
skf = StratifiedKFold(n_splits=10)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=skf)
    print(f"{name}  Accuracy: {np.mean(scores):.4f}")

In [ ]:
#SOBEL

from sklearn.preprocessing import LabelEncoder

train_data = []
train_labels = []

image_folder = '/content/drive/MyDrive/DATASET_PROJECT_DIP'
train_dir = os.path.join(image_folder, "training_set")
for class_name in os.listdir(train_dir):
  class_dir = os.path.join(train_dir, class_name)
  for image_name in os.listdir(class_dir):
      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      img_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5)
      img_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)
      img_sobel = np.sqrt(img_x**2 + img_y**2)
      (h, w) = img_sobel.shape[:2]
      center = (w / 2, h / 2)
      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(img_sobel, M, (w, h), borderValue=(0,0,0))

          train_data.append(rotated_image.flatten())
          train_labels.append(class_name)

train_data = np.array(train_data)
train_labels = np.array(train_labels)


validation_data = []
validation_labels = []
validation_dir = os.path.join(image_folder, "validation_set")
for class_name in os.listdir(validation_dir):
  class_dir = os.path.join(validation_dir, class_name)
  for image_name in os.listdir(class_dir):

      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      #sobel
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      img_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5)
      img_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)
      img_sobel = np.sqrt(img_x**2 + img_y**2)
      (h, w) = img_sobel.shape[:2]
      center = (w / 2, h / 2)
      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(img_sobel, M, (w, h), borderValue=(0,0,0))

          validation_data.append(rotated_image.flatten())
          validation_labels.append(class_name)

validation_data = np.array(validation_data)
validation_labels = np.array(validation_labels)

# Encode labels to numerical format
le = LabelEncoder()
train_labels_encoded = le.fit_transform(train_labels)
validation_labels_encoded = le.transform(validation_labels)


knn_classifier = KNeighborsClassifier(n_neighbors = 1,metric="manhattan")
knn_classifier.fit(train_data, train_labels_encoded)
predictions = knn_classifier.predict(validation_data)
accuracy = accuracy_score(validation_labels_encoded, predictions)
print(f"Accuracy: {accuracy}")


from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score, RepeatedStratifiedKFold, StratifiedKFold, LeaveOneOut
from sklearn.neural_network import MLPClassifier
from sklearn import svm
from sklearn.preprocessing import LabelEncoder
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os



#menggunakan gabungan train dan validation set
X = np.vstack((train_data, validation_data))
y = np.concatenate((train_labels, validation_labels))

# Encode labels to numerical format if not already done
le = LabelEncoder()
y_encoded = le.fit_transform(y)


# Split data into training, validation, and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

# Initialize and train the KNN classifier
knn = KNeighborsClassifier(n_neighbors= 2) # neighbors
knn.fit(X_train, y_train)

# Make predictions
y_pred = knn.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

#train validation
print("\nTrain Validation")
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  clf.fit(X_train, y_train)
  output=clf.predict(X_val)
  acc = 100*accuracy_score(y_val, output)
  print('k = %g, test accuracy = %g%%'%(k,acc))
clf = KNeighborsClassifier(n_neighbors=1)
clf.fit(X_train, y_train)
output = clf.predict(X_test)
# The accuracy should be evaluated against the test set, not the validation set
acc = 100*accuracy_score(y_test, output)
print('k = %g, test accuracy = %g%%'%(1, acc))

#stratified repeated hold out
print("\nstratified repeated hold out")
sss = list(StratifiedShuffleSplit(n_splits=10, test_size=0.333, random_state=1).split(X, y_encoded))
mean_acc = []
acc = 0
K = 1
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  # Use X and y_encoded for cross_val_score with the defined sss splits
  accuracy = cross_val_score(clf, X, y_encoded, cv=sss)
  mean_acc.append(100*np.mean(accuracy))
  if(100*accuracy.mean()>acc):
    acc = 100*accuracy.mean()
    K = k
print('highest mean accuracy is at k = %g, test accuracy = %g%%'%(K,acc))
plt.plot(range(1,11),mean_acc,'o-')
plt.title('KNN Accuracy with Stratified Repeated Hold Out')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Mean Accuracy (%)')
plt.show()

#10-fold cross validation
print("\n 10-fold cross validation")
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=10)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

#10-fold stratified validation
print("\n stratified validation")
skf = StratifiedKFold(n_splits=10)
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=skf)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

# leave one out
print("\n leave one out")
loo = LeaveOneOut()
for k in range(1,11):
  clf = KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=loo)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

# Evaluate other classifiers
ann = MLPClassifier(max_iter=1000, random_state=42, activation = 'relu')
svm_model = svm.SVC(random_state=42) # Instantiate SVM classifier

# 1. Train-test split
print("\nTrain-Test Split")
xtrain, xtest, ytrain, ytest = train_test_split(X, y_encoded, test_size=0.3, stratify=y_encoded)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    model.fit(xtrain, ytrain)
    ypred = model.predict(xtest)
    acc = accuracy_score(ytest, ypred)
    print(f"{name} Accuracy: {acc:.4f}")

# 2. Stratified repeated
print("\n Stratified Repeated")
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=rskf)
    print(f"{name}  Accuracy: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# 3.  Cross Validation
print("\n cross")
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=10)
    print(f"{name}  Accuracy: {np.mean(scores):.4f}")

# 4. Stratified
print("\n Stratified")
skf = StratifiedKFold(n_splits=10)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=skf)
    print(f"{name}  Accuracy: {np.mean(scores):.4f}")



In [ ]:
from sklearn.preprocessing import LabelEncoder
#tanpa preprocess tidak dipakai
train_data = []
train_labels = []

image_folder = '/content/drive/MyDrive/DATASET_PROJECT_DIP'
train_dir = os.path.join(image_folder, "training_set")
for class_name in os.listdir(train_dir):
  class_dir = os.path.join(train_dir, class_name)
  for image_name in os.listdir(class_dir):
      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      (h, w) = image.shape[:2]
      center = (w / 2, h / 2)
      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(image, M, (w, h), borderValue=(0,0,0))

          train_data.append(rotated_image.flatten())
          train_labels.append(class_name)

train_data = np.array(train_data)
train_labels = np.array(train_labels)


validation_data = []
validation_labels = []
validation_dir = os.path.join(image_folder, "validation_set")
for class_name in os.listdir(validation_dir):
  class_dir = os.path.join(validation_dir, class_name)
  for image_name in os.listdir(class_dir):

      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      (h, w) = image.shape[:2]
      center = (w / 2, h / 2)
      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(image, M, (w, h), borderValue=(0,0,0))

          validation_data.append(rotated_image.flatten())
          validation_labels.append(class_name)

validation_data = np.array(validation_data)
validation_labels = np.array(validation_labels)

# Encode labels to numerical format
le = LabelEncoder()
train_labels_encoded = le.fit_transform(train_labels)
validation_labels_encoded = le.transform(validation_labels)


knn_classifier = KNeighborsClassifier(n_neighbors = 2,metric="manhattan")
knn_classifier.fit(train_data, train_labels_encoded)
predictions = knn_classifier.predict(validation_data)
accuracy = accuracy_score(validation_labels_encoded, predictions)
print(f"Accuracy: {accuracy}")




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#LAPLACIAN

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix # Added import

train_data = []
train_labels = []

image_folder = '/content/drive/MyDrive/DATASET_PROJECT_DIP'
train_dir = os.path.join(image_folder, "training_set")
for class_name in os.listdir(train_dir):
  class_dir = os.path.join(train_dir, class_name)
  for image_name in os.listdir(class_dir):
      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      gray = cv2.GaussianBlur(gray, (3,3), 0)

      laplacian = cv2.Laplacian(gray,cv2.CV_64F, ksize = 3)
      enchanced_img = cv2.convertScaleAbs(gray - laplacian)
      (h, w) = image.shape[:2]
      center = (w / 2, h / 2)
      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(enchanced_img, M, (w, h), borderValue=(0,0,0))

          train_data.append(rotated_image.flatten())
          train_labels.append(class_name)

train_data = np.array(train_data)
train_labels = np.array(train_labels)


validation_data = []
validation_labels = []
validation_dir = os.path.join(image_folder, "validation_set")
for class_name in os.listdir(validation_dir):
  class_dir = os.path.join(validation_dir, class_name)
  for image_name in os.listdir(class_dir):

      image_path = os.path.join(class_dir, image_name)
      image = cv2.imread(image_path)
      if(image is None):
        continue
      image = cv2.resize(image, (64,64), interpolation = cv2.INTER_AREA)
      gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      gray = cv2.GaussianBlur(gray, (3,3), 0)
      laplacian = cv2.Laplacian(gray,cv2.CV_64F, ksize = 3)
      enchanced_img = cv2.convertScaleAbs(gray - laplacian)
      (h, w) = image.shape[:2]
      center = (w / 2, h / 2)
      for angle in range(0, 360, 45):
          M = cv2.getRotationMatrix2D(center, angle, 1.0)
          rotated_image = cv2.warpAffine(enchanced_img, M, (w, h), borderValue=(0,0,0))

          validation_data.append(rotated_image.flatten())
          validation_labels.append(class_name)

validation_data = np.array(validation_data)
validation_labels = np.array(validation_labels)

# Encode labels to numerical format
le = LabelEncoder()
train_labels_encoded = le.fit_transform(train_labels)
validation_labels_encoded = le.transform(validation_labels)

#knn classifier
knn_classifier = KNeighborsClassifier(n_neighbors = 3,metric="manhattan")
knn_classifier.fit(train_data, train_labels_encoded)
predictions = knn_classifier.predict(validation_data)
accuracy = accuracy_score(validation_labels_encoded, predictions)
print(f"Accuracy: {accuracy}")

#svm classifier
svm_classifier = svm.SVC(kernel="sigmoid")
svm_classifier.fit(train_data, train_labels)
predictions = svm_classifier.predict(validation_data)
accuracy = accuracy_score(validation_labels, predictions)
print(f"SVM Sigmoid Accuracy: {accuracy}")
svm_classifier = svm.SVC(kernel="linear")
svm_classifier.fit(train_data, train_labels)
predictions = svm_classifier.predict(validation_data)
accuracy = accuracy_score(validation_labels, predictions)
print(f"SVM Linear Accuracy: {accuracy}")

#confusion matrix
clf = KNeighborsClassifier(n_neighbors=5)
clf.fit(train_data, train_labels)
output = clf.predict(validation_data)
confusion_matrix(validation_labels, output)

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score, RepeatedStratifiedKFold, StratifiedKFold, LeaveOneOut
from sklearn.neural_network import MLPClassifier
from sklearn import svm
from sklearn.preprocessing import LabelEncoder
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os



#menggunakan gabungan train dan validation set
X = np.vstack((train_data, validation_data))
y = np.concatenate((train_labels, validation_labels))

# Encode labels to numerical format if not already done
le = LabelEncoder()
y_encoded = le.fit_transform(y)


# Split data ke training, validation, dan testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

# Initialize and train the KNN classifier
knn = KNeighborsClassifier(n_neighbors= 2) # neighbors
knn.fit(X_train, y_train)

# Make predictions
y_pred = knn.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

#train validation
print("\nTrain Validation")
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  clf.fit(X_train, y_train)
  output=clf.predict(X_val)
  acc = 100*accuracy_score(y_val, output)
  print('k = %g, test accuracy = %g%%'%(k,acc))
clf = KNeighborsClassifier(n_neighbors=1)
clf.fit(X_train, y_train)
output = clf.predict(X_test)
# The accuracy should be evaluated against the test set, not the validation set
acc = 100*accuracy_score(y_test, output)
print('k = %g, test accuracy = %g%%'%(1, acc))

#stratified repeated hold out
print("\nstratified repeated hold out")
sss = list(StratifiedShuffleSplit(n_splits=10, test_size=0.333, random_state=1).split(X, y_encoded))
mean_acc = []
acc = 0
K = 1
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  # Use X and y_encoded for cross_val_score with the defined sss splits
  accuracy = cross_val_score(clf, X, y_encoded, cv=sss)
  mean_acc.append(100*np.mean(accuracy))
  if(100*accuracy.mean()>acc):
    acc = 100*accuracy.mean()
    K = k
print('highest mean accuracy is at k = %g, test accuracy = %g%%'%(K,acc))
plt.plot(range(1,11),mean_acc,'o-')
plt.title('KNN Accuracy with Stratified Repeated Hold Out')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Mean Accuracy (%)')
plt.show()

#10-fold cross validation
print("\n 10-fold cross validation")
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=10)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

#10-fold stratified validation
print("\n stratified validation")
skf = StratifiedKFold(n_splits=10)
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=skf)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

# leave one out
print("\n leave one out")
loo = LeaveOneOut()
for k in range(1,11):
  clf = KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=loo)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

# Evaluate other classifiers
ann = MLPClassifier(max_iter=1000, random_state=42, activation='relu')
svm_model = svm.SVC(kernel = 'linear',random_state=42)

# 1. Train-test split
print("\nTrain-Test Split")
xtrain, xtest, ytrain, ytest = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    model.fit(xtrain, ytrain)
    ypred = model.predict(xtest)
    acc = accuracy_score(ytest, ypred)
    print(f"{name} Accuracy: {acc:.4f}")

# 2. Stratified repeated
print("\n Stratified Repeated")
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=rskf)
    print(f"{name}  Accuracy: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# 3.  Cross Validation
print("\n cross")
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=10)
    print(f"{name}  Accuracy: {np.mean(scores):.4f}")

# 4. Stratified
print("\n Stratified")
skf = StratifiedKFold(n_splits=10)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=skf)
    print(f"{name}  Accuracy: {np.mean(scores):.4f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score, RepeatedStratifiedKFold, StratifiedKFold, LeaveOneOut
from sklearn.neural_network import MLPClassifier
from sklearn import svm
from sklearn.preprocessing import LabelEncoder
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os



#menggunakan gabungan train dan validation set
X = np.vstack((train_data, validation_data))
y = np.concatenate((train_labels, validation_labels))

# Encode labels to numerical format if not already done
le = LabelEncoder()
y_encoded = le.fit_transform(y)


# Split data into training, validation, and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

# Initialize and train the KNN classifier
knn = KNeighborsClassifier(n_neighbors= 2) # neighbors
knn.fit(X_train, y_train)

# Make predictions
y_pred = knn.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

#train validation
print("\nTrain Validation")
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  clf.fit(X_train, y_train)
  output=clf.predict(X_val)
  acc = 100*accuracy_score(y_val, output)
  print('k = %g, test accuracy = %g%%'%(k,acc))
clf = KNeighborsClassifier(n_neighbors=1)
clf.fit(X_train, y_train)
output = clf.predict(X_test)
# The accuracy should be evaluated against the test set, not the validation set
acc = 100*accuracy_score(y_test, output)
print('k = %g, test accuracy = %g%%'%(1, acc))

#stratified repeated hold out
print("\nstratified repeated hold out")
sss = list(StratifiedShuffleSplit(n_splits=10, test_size=0.333, random_state=1).split(X, y_encoded))
mean_acc = []
acc = 0
K = 1
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  # Use X and y_encoded for cross_val_score with the defined sss splits
  accuracy = cross_val_score(clf, X, y_encoded, cv=sss)
  mean_acc.append(100*np.mean(accuracy))
  if(100*accuracy.mean()>acc):
    acc = 100*accuracy.mean()
    K = k
print('highest mean accuracy is at k = %g, test accuracy = %g%%'%(K,acc))
plt.plot(range(1,11),mean_acc,'o-')
plt.title('KNN Accuracy with Stratified Repeated Hold Out')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Mean Accuracy (%)')
plt.show()

#10-fold cross validation
print("\n 10-fold cross validation")
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=10)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

#10-fold stratified validation
print("\n stratified validation")
skf = StratifiedKFold(n_splits=10)
for k in range(1,11):
  clf=KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=skf)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

# leave one out
print("\n leave one out")
loo = LeaveOneOut()
for k in range(1,11):
  clf = KNeighborsClassifier(n_neighbors=k)
  accuracy = cross_val_score(clf, X, y_encoded, cv=loo)
  print('k = %g, test accuracy = %g%%'%(k,100*accuracy.mean()))

# Evaluate other classifiers
ann = MLPClassifier(max_iter=1000, random_state=42)
svm_model = svm.SVC(random_state=42) # Instantiate SVM classifier

# 1. Train-test split
print("\nTrain-Test Split")
xtrain, xtest, ytrain, ytest = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    model.fit(xtrain, ytrain)
    ypred = model.predict(xtest)
    acc = accuracy_score(ytest, ypred)
    print(f"{name} Accuracy: {acc:.4f}")

# 2. Stratified repeated
print("\n Stratified Repeated")
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=rskf)
    print(f"{name}  Accuracy: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# 3.  Cross Validation
print("\n cross")
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=10)
    print(f"{name}  Accuracy: {np.mean(scores):.4f}")

# 4. Stratified
print("\n Stratified")
skf = StratifiedKFold(n_splits=10)
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=skf)
    print(f"{name}  Accuracy: {np.mean(scores):.4f}")

# 5. Leave-One-Out CV
print("\n Leave-One-Out")
loo = LeaveOneOut()
for model, name in [(ann, "ANN"), (svm_model, "SVM")]:
    scores = cross_val_score(model, X, y_encoded, cv=loo)
    print(f"{name} Accuracy: {np.mean(scores):.4f}")